In [9]:
import sys
from pathlib import Path

import pandas as pd

# Add project root so imports like `from src...` work in notebooks/
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.run import Config
from src.grid import grid_search

import wandb

In [ ]:
#The range of parameters to search over
grid = {
    "mr_window": [10, 20, 30, 50],
    "vol_q": [0.1, 0.7, 0.8],
    "fee_bps": [0.5, 1.0, 2.0],
}

symbols = ["SPY", "2330.TW"]
base_cfg = Config()

res = grid_search(symbols=symbols, base_cfg=base_cfg, grid=grid)
res.to_csv("reports/grid_results.csv", index=False)
res.head()


,Symbol,Strategy,FinalEquity,AnnRet,AnnVol,Sharpe,MaxDD,Calmar,param_mr_window,param_vol_q,param_fee_bps
0,SPY,Buy&Hold,6.572025,0.136310,0.170940,0.833541,-2.255836,0.060426,10,0.1,0.5
1,SPY,Momentum,0.951758,-0.003350,0.090008,0.008328,-1.511198,-0.002217,10,0.1,0.5
2,SPY,Mean Reversion,2.470379,0.063302,0.135141,0.522099,-1.844776,0.034314,10,0.1,0.5
3,SPY,Mean Reversion with Regime,1.285656,0.017200,0.043208,0.416253,-1.108244,0.015520,10,0.1,0.5
4,2330.TW,Buy&Hold,27.586594,0.262697,0.252545,1.049983,-8.398519,0.031279,10,0.1,0.5


In [5]:
res = pd.read_csv("reports/grid_results.csv")
res.columns

Index(['Symbol', 'Strategy', 'FinalEquity', 'AnnRet', 'AnnVol', 'Sharpe',
       'MaxDD', 'Calmar', 'param_mr_window', 'param_vol_q', 'param_fee_bps'],
      dtype='str')

In [6]:
res["Strategy"].unique()

<ArrowStringArray>
['Buy&Hold', 'Momentum', 'Mean Reversion', 'Mean Reversion with Regime']
Length: 4, dtype: str

In [8]:
res = pd.read_csv("reports/grid_results.csv")

res["Strategy"] = res["Strategy"].astype(str).str.strip()
target = res[res["Strategy"] == "Mean Reversion with Regime"].copy()

best = (
    target.sort_values(["Symbol", "Sharpe"], ascending=[True, False])
    .groupby("Symbol")
    .head(5)
)

best[["Symbol", "Strategy", "Sharpe", "MaxDD", "Calmar", "param_mr_window", 
      "param_vol_q", "param_fee_bps"]]

,Symbol,Strategy,Sharpe,MaxDD,Calmar,param_mr_window,param_vol_q,param_fee_bps
79,2330.TW,Mean Reversion with Regime,1.038133,-1.107023,0.047560,20,0.1,0.5
87,2330.TW,Mean Reversion with Regime,1.030233,-1.106671,0.047181,20,0.1,1.0
95,2330.TW,Mean Reversion with Regime,1.014395,-1.105967,0.046424,20,0.1,2.0
151,2330.TW,Mean Reversion with Regime,0.949066,-1.100030,0.043303,30,0.1,0.5
159,2330.TW,Mean Reversion with Regime,0.942943,-1.100334,0.042984,30,0.1,1.0
267,SPY,Mean Reversion with Regime,0.882145,-1.236003,0.052289,50,0.8,0.5
275,SPY,Mean Reversion with Regime,0.871634,-1.235057,0.051657,50,0.8,1.0
243,SPY,Mean Reversion with Regime,0.865867,-1.163009,0.047285,50,0.7,0.5
251,SPY,Mean Reversion with Regime,0.854939,-1.161380,0.046711,50,0.7,1.0
283,SPY,Mean Reversion with Regime,0.850594,-1.233157,0.050391,50,0.8,2.0


In [10]:
wandb.init(project="quant-regime-project", name="grid-search")
wandb.log({"grid_results": wandb.Table(dataframe=res)})

#We can also put the best results into a table and log it
wandb.log({"best_mr_results": wandb.Table(dataframe=best)})

wandb.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/luchengliang/.netrc.
wandb: Currently logged in as: lianglu3366 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: ERROR The nbformat package was not found. It is required to save notebook history.
